In [0]:
import requests, io, csv

EIA_KEY = dbutils.secrets.get("oil-shock", "eia-api-key")
GEMINI_KEY = dbutils.secrets.get("oil-shock", "gemini-api-key")
TIMEOUT = 60
results = []

def clean(text):
    # Never show keys in output
    return str(text).replace(EIA_KEY, "***").replace(GEMINI_KEY, "***")[:200]

def check(name, fn):
    try:
        detail = fn()
        results.append((name, "OK", detail))
    except Exception as e:
        results.append((name, "FAILED", clean(f"{type(e).__name__}: {e}")))

# 1. EIA API: latest Brent spot prices
def eia_api():
    r = requests.get(
        "https://api.eia.gov/v2/petroleum/pri/spt/data/",
        params={
            "api_key": EIA_KEY, "frequency": "daily", "data[0]": "value",
            "facets[series][]": "RBRTE",
            "sort[0][column]": "period", "sort[0][direction]": "desc", "length": 3,
        },
        timeout=TIMEOUT,
    )
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}")
    rows = r.json()["response"]["data"]
    return "Brent spot: " + ", ".join(f"{x['period']} = ${x['value']}" for x in rows)

# 2. EIA bulk files (no key)
def eia_bulk():
    r = requests.get("https://www.eia.gov/opendata/bulk/manifest.txt", timeout=TIMEOUT)
    r.raise_for_status()
    return f"Manifest downloaded ({len(r.content):,} bytes)"

# 3. JODI 2026 CSV (read only the first lines)
def jodi():
    url = "https://www.jodidata.org/_resources/files/downloads/oil-data/annual-csv/primary/primaryyear2026.csv"
    with requests.get(url, stream=True, timeout=TIMEOUT) as r:
        r.raise_for_status()
        lines = []
        for line in r.iter_lines(decode_unicode=True):
            lines.append(line)
            if len(lines) >= 3:
                break
    return "Header: " + lines[0][:150]

# 4. IMF PortWatch: latest Hormuz transits
def portwatch():
    r = requests.get(
        "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/arcgis/rest/services/Daily_Chokepoints_Data/FeatureServer/0/query",
        params={
            "where": "portid='chokepoint6'", "outFields": "date,n_total,n_tanker",
            "orderByFields": "date DESC", "resultRecordCount": 3, "f": "json",
        },
        timeout=TIMEOUT,
    )
    r.raise_for_status()
    feats = [f["attributes"] for f in r.json()["features"]]
    return "Hormuz: " + ", ".join(f"{a['date']} total={a['n_total']} tankers={a['n_tanker']}" for a in feats)

# 5. CFTC trader positions: WTI crude (disaggregated, futures only)
def cftc():
    r = requests.get(
        "https://publicreporting.cftc.gov/resource/72hh-3qpy.json",
        params={
            "$where": "market_and_exchange_names like '%CRUDE OIL, LIGHT SWEET%'",
            "$order": "report_date_as_yyyy_mm_dd DESC", "$limit": 2,
        },
        timeout=TIMEOUT,
    )
    r.raise_for_status()
    rows = r.json()
    if not rows:
        return "Reached CFTC, but no rows matched (filter to adjust later)"
    x = rows[0]
    return (f"{x.get('report_date_as_yyyy_mm_dd','')[:10]} | {x.get('market_and_exchange_names','')[:45]} | "
            f"money managers long={x.get('m_money_positions_long_all')} short={x.get('m_money_positions_short_all')}")

# 6. GDELT: latest 15-minute file list
def gdelt():
    for url in ["https://data.gdeltproject.org/gdeltv2/lastupdate.txt",
                "http://data.gdeltproject.org/gdeltv2/lastupdate.txt"]:
        try:
            r = requests.get(url, timeout=TIMEOUT)
            r.raise_for_status()
            first = r.text.strip().splitlines()[0].split()[-1]
            return f"Latest file: {first.rsplit('/', 1)[-1]} (via {url.split(':')[0]})"
        except Exception:
            continue
    raise RuntimeError("Both https and http failed")

# 7. Gemini: label one sample event
def gemini():
    base = "https://generativelanguage.googleapis.com/v1beta"
    headers = {"x-goog-api-key": GEMINI_KEY, "Content-Type": "application/json"}
    body = {"contents": [{"parts": [{"text":
        "Label with ONE category: vessel attack, infrastructure strike, chokepoint closure, "
        "sanctions, talks, OPEC+ decision, other. Reply with the category only.\n\n"
        "Event: A tanker was hit by a drone near the Strait of Hormuz."}]}]}
    errors = []
    for model in ["models/gemini-3-flash-preview", "models/gemini-flash-latest", "models/gemini-2.5-flash"]:
        r = requests.post(f"{base}/{model}:generateContent", headers=headers, json=body, timeout=TIMEOUT)
        if r.ok:
            answer = r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
            return f"{model.split('/')[-1]} -> {answer}"
        errors.append(f"{model.split('/')[-1]}: HTTP {r.status_code}")
    raise RuntimeError("; ".join(errors))

check("1 EIA API", eia_api)
check("2 EIA bulk files", eia_bulk)
check("3 JODI", jodi)
check("4 PortWatch", portwatch)
check("5 CFTC", cftc)
check("6 GDELT", gdelt)
check("7 Gemini", gemini)

print(f"{'Check':<18} {'Result':<8} Detail")
print("-" * 110)
for name, status, detail in results:
    print(f"{name:<18} {status:<8} {detail}")
print("-" * 110)
print(f"{sum(s == 'OK' for _, s, _ in results)} of {len(results)} checks passed")

In [0]:
import requests

def jodi():
    url = "https://www.jodidata.org/_resources/files/downloads/oil-data/annual-csv/primary/primaryyear2026.csv"
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        lines = []
        for raw in r.iter_lines():
            lines.append(raw.decode("utf-8", errors="replace"))
            if len(lines) >= 3:
                break
    print("JODI OK")
    for line in lines:
        print(line[:150])

jodi()

In [0]:
import requests, re

TIMEOUT = 60
results = []

def check(name, fn):
    try:
        results.append((name, "OK", fn()))
    except Exception as e:
        results.append((name, "FAILED", f"{type(e).__name__}: {str(e)[:150]}"))

# 1. NOAA: active Atlantic storms right now
def noaa_current():
    r = requests.get("https://www.nhc.noaa.gov/CurrentStorms.json", timeout=TIMEOUT)
    r.raise_for_status()
    storms = r.json().get("activeStorms", [])
    if not storms:
        return "Reached NOAA. No active storms right now."
    return "Active storms: " + ", ".join(f"{s.get('name')} ({s.get('classification')})" for s in storms[:5])

# 2. NOAA: historical storm tracks file list (HURDAT2)
def noaa_history():
    r = requests.get("https://www.nhc.noaa.gov/data/hurdat/", timeout=TIMEOUT)
    r.raise_for_status()
    files = sorted(set(re.findall(r'hurdat2-1851-[\w\-]+\.txt', r.text)))
    return f"Atlantic history files found: {len(files)}. Latest: {files[-1] if files else 'none'}"

# 3. OECD: leading economic indicators (latest value)
def oecd():
    base = "https://sdmx.oecd.org/public/rest/data/OECD.SDD.STES,DSD_STES@DF_CLI,"
    for key in ["USA+CHN+G20.M.LI...AA...H", "all"]:
        r = requests.get(f"{base}/{key}",
                         params={"lastNObservations": 1, "format": "csvfilewithlabels"},
                         timeout=TIMEOUT)
        if r.ok and len(r.text.splitlines()) > 1:
            lines = r.text.splitlines()
            return f"Rows: {len(lines) - 1} (query '{key[:25]}'). First row: {lines[1][:120]}"
    raise RuntimeError(f"HTTP {r.status_code}")

# 4. BSEE: news releases page (shut-in figures are published here per storm)
def bsee():
    r = requests.get("https://www.bsee.gov/newsroom/latest-news", timeout=TIMEOUT,
                     headers={"User-Agent": "Mozilla/5.0 (portfolio project)"})
    r.raise_for_status()
    hits = len(re.findall(r'Activity Statistics', r.text))
    return f"Reached BSEE news page. 'Activity Statistics' mentions on first page: {hits}"

check("1 NOAA active storms", noaa_current)
check("2 NOAA storm history", noaa_history)
check("3 OECD indicators", oecd)
check("4 BSEE news", bsee)

print(f"{'Check':<22} {'Result':<8} Detail")
print("-" * 110)
for name, status, detail in results:
    print(f"{name:<22} {status:<8} {detail}")
print("-" * 110)
print(f"{sum(s == 'OK' for _, s, _ in results)} of {len(results)} checks passed")

In [0]:
import requests, re, io, csv
import xml.etree.ElementTree as ET

TIMEOUT = 60
UA = {"User-Agent": "Mozilla/5.0 (portfolio project)"}
results = []

def check(name, fn):
    try:
        results.append((name, "OK", fn()))
    except Exception as e:
        results.append((name, "FAILED", f"{type(e).__name__}: {str(e)[:150]}"))

# 1. OPEC basket price (official daily XML)
def opec_basket():
    r = requests.get("https://www.opec.org/basket/basketDayArchives.xml", headers=UA, timeout=TIMEOUT)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    rows = [(el.attrib.get("data"), el.attrib.get("val")) for el in root.iter() if el.attrib.get("val")]
    rows = [x for x in rows if x[0]]
    rows.sort(key=lambda x: x[0])
    return f"{len(rows):,} daily values. Latest: " + ", ".join(f"{d} = ${v}" for d, v in rows[-3:])

# 2. ICE Brent trader positions (weekly CSV)
def ice_cot():
    for year in [2026, 2025]:
        url = f"https://www.ice.com/publicdocs/futures/COTHist{year}.csv"
        r = requests.get(url, headers=UA, timeout=TIMEOUT)
        if r.ok and len(r.text) > 1000:
            reader = csv.reader(io.StringIO(r.text))
            header = next(reader)
            brent = [row for row in reader if row and "Brent" in row[0]]
            latest = brent[0][:3] if brent else []
            return (f"File COTHist{year}.csv found. Columns: {len(header)}. "
                    f"Brent rows: {len(brent)}. First Brent row: {latest}")
    raise RuntimeError(f"HTTP {r.status_code}")

check("1 OPEC basket", opec_basket)
check("2 ICE Brent COT", ice_cot)

print(f"{'Check':<18} {'Result':<8} Detail")
print("-" * 110)
for name, status, detail in results:
    print(f"{name:<18} {status:<8} {detail}")